In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# MIMIC-IV SAE LABEL + PATIENT-LEVEL SPLIT
# ============================================================

ROOT = Path("/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning")

INPUT = (
    ROOT
    / "sae_mimiciv_v2"
    / "data"
    / "mimiciv_sae_hourly_features.csv"
)

OUT = ROOT / "sae_mimiciv_v2" / "data"
OUT.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. LOAD HOURLY FEATURE TABLE
# ============================================================

df = pd.read_csv(INPUT)

print("Input shape:", df.shape)
print("Patients:", df["subject_id"].nunique())
print("ICU stays:", df["stay_id"].nunique())

# ============================================================
# 2. LOAD SAE COHORT AGAIN
# ============================================================

HOSP = ROOT / "data" / "raw" / "mimic-iv" / "hosp"
ICU = ROOT / "data" / "raw" / "mimic-iv" / "icu"

diagnoses = pd.read_csv(
    HOSP / "diagnoses_icd.csv.gz"
)

d_icd = pd.read_csv(
    HOSP / "d_icd_diagnoses.csv.gz"
)

icustays = pd.read_csv(
    ICU / "icustays.csv.gz"
)

# ============================================================
# 3. FIND SEPSIS + ENCEPHALOPATHY DIAGNOSES
# ============================================================

text_cols = [
    c for c in ["long_title", "short_title"]
    if c in d_icd.columns
]

sepsis_mask = False

for c in text_cols:
    sepsis_mask = sepsis_mask | d_icd[c].astype(str).str.contains(
        "sepsis|septic",
        case=False,
        na=False,
        regex=True
    )

enceph_mask = False

for c in text_cols:
    enceph_mask = enceph_mask | d_icd[c].astype(str).str.contains(
        "encephal",
        case=False,
        na=False,
        regex=True
    )

sepsis_codes = d_icd.loc[
    sepsis_mask,
    ["icd_code", "icd_version"]
].drop_duplicates()

enceph_codes = d_icd.loc[
    enceph_mask,
    ["icd_code", "icd_version"]
].drop_duplicates()

sepsis_dx = diagnoses.merge(
    sepsis_codes,
    on=["icd_code", "icd_version"],
    how="inner"
)

enceph_dx = diagnoses.merge(
    enceph_codes,
    on=["icd_code", "icd_version"],
    how="inner"
)

# SAE = admission has BOTH sepsis and encephalopathy
sae_hadm = set(
    sepsis_dx["hadm_id"]
).intersection(
    set(enceph_dx["hadm_id"])
)

print("\nSAE admissions:", len(sae_hadm))

# ============================================================
# 4. CREATE SAE EVENT LABEL
# ============================================================

# All hours belonging to an SAE ICU stay
df["sae"] = (
    df["hadm_id"]
    .isin(sae_hadm)
    .astype(int)
)

print("\nSAE label distribution:")
print(df["sae"].value_counts())

# ============================================================
# 5. DETERMINE SAE ONSET / EVENT HOUR
# ============================================================

# For each SAE ICU stay, use the earliest available ICU hour
# as the event onset proxy because diagnosis data does not provide
# an exact clinical onset timestamp.

sae_stays = df.loc[
    df["sae"] == 1,
    ["subject_id", "hadm_id", "stay_id", "hour"]
].copy()

if len(sae_stays) > 0:

    onset = (
        sae_stays
        .groupby("stay_id", as_index=False)["hour"]
        .min()
        .rename(columns={"hour": "sae_onset_hour"})
    )

    df = df.merge(
        onset,
        on="stay_id",
        how="left"
    )

else:

    df["sae_onset_hour"] = np.nan

# ============================================================
# 6. FUTURE SAE TARGET
# ============================================================

# Predict SAE in the next 6 hours.
#
# For an SAE stay, hours immediately preceding the onset
# become positive prediction examples.

HORIZON = 6

df["future_sae_6h"] = 0

mask = (
    df["sae_onset_hour"].notna()
    &
    (df["hour"] < df["sae_onset_hour"])
    &
    (
        df["hour"]
        >=
        df["sae_onset_hour"] - HORIZON
    )
)

df.loc[mask, "future_sae_6h"] = 1

print("\nFuture SAE distribution:")
print(
    df["future_sae_6h"].value_counts()
)

# ============================================================
# 7. REMOVE CURRENT-EVENT HOURS FROM PREDICTION DATA
# ============================================================

# We want prediction BEFORE the event, not after the diagnosis.

model_df = df[
    df["hour"] < df["sae_onset_hour"].fillna(np.inf)
].copy()

print("\nPrediction dataset:", model_df.shape)

print(
    "Positive future-SAE rows:",
    model_df["future_sae_6h"].sum()
)

# ============================================================
# 8. PATIENT-LEVEL SPLIT
# ============================================================

patients = (
    model_df["subject_id"]
    .drop_duplicates()
    .sample(
        frac=1,
        random_state=42
    )
    .tolist()
)

n_patients = len(patients)

n_train = int(
    n_patients * 0.70
)

n_val = int(
    n_patients * 0.15
)

train_patients = set(
    patients[:n_train]
)

val_patients = set(
    patients[n_train:n_train + n_val]
)

test_patients = set(
    patients[n_train + n_val:]
)

train = model_df[
    model_df["subject_id"].isin(train_patients)
].copy()

validation = model_df[
    model_df["subject_id"].isin(val_patients)
].copy()

test = model_df[
    model_df["subject_id"].isin(test_patients)
].copy()

# ============================================================
# 9. VERIFY NO PATIENT LEAKAGE
# ============================================================

train_set = set(train["subject_id"])
val_set = set(validation["subject_id"])
test_set = set(test["subject_id"])

assert len(train_set & val_set) == 0
assert len(train_set & test_set) == 0
assert len(val_set & test_set) == 0

print("\nPATIENT LEAKAGE CHECK: PASSED")

# ============================================================
# 10. SAVE SPLITS
# ============================================================

train_file = OUT / "train_sae.parquet"
val_file = OUT / "validation_sae.parquet"
test_file = OUT / "test_sae.parquet"

train.to_parquet(
    train_file,
    index=False
)

validation.to_parquet(
    val_file,
    index=False
)

test.to_parquet(
    test_file,
    index=False
)

# ============================================================
# 11. SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("SAE DATASET READY")
print("=" * 60)

print(
    f"TOTAL patients:        {model_df.subject_id.nunique()}"
)
print(
    f"TOTAL ICU stays:       {model_df.stay_id.nunique()}"
)
print(
    f"TOTAL rows:            {len(model_df)}"
)

print("\nTRAIN")
print(
    "Patients:",
    train.subject_id.nunique()
)
print(
    "ICU stays:",
    train.stay_id.nunique()
)
print(
    "Rows:",
    len(train)
)
print(
    "Positive future SAE:",
    int(train.future_sae_6h.sum())
)

print("\nVALIDATION")
print(
    "Patients:",
    validation.subject_id.nunique()
)
print(
    "ICU stays:",
    validation.stay_id.nunique()
)
print(
    "Rows:",
    len(validation)
)
print(
    "Positive future SAE:",
    int(validation.future_sae_6h.sum())
)

print("\nTEST")
print(
    "Patients:",
    test.subject_id.nunique()
)
print(
    "ICU stays:",
    test.stay_id.nunique()
)
print(
    "Rows:",
    len(test)
)
print(
    "Positive future SAE:",
    int(test.future_sae_6h.sum())
)

print("\nSAVED FILES:")
print(train_file)
print(val_file)
print(test_file)

Input shape: (1517, 50)
Patients: 6
ICU stays: 9

SAE admissions: 6

SAE label distribution:
sae
1    1517
Name: count, dtype: int64

Future SAE distribution:
future_sae_6h
0    1517
Name: count, dtype: int64

Prediction dataset: (0, 53)
Positive future-SAE rows: 0

PATIENT LEAKAGE CHECK: PASSED

SAE DATASET READY
TOTAL patients:        0
TOTAL ICU stays:       0
TOTAL rows:            0

TRAIN
Patients: 0
ICU stays: 0
Rows: 0
Positive future SAE: 0

VALIDATION
Patients: 0
ICU stays: 0
Rows: 0
Positive future SAE: 0

TEST
Patients: 0
ICU stays: 0
Rows: 0
Positive future SAE: 0

SAVED FILES:
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_mimiciv_v2/data/train_sae.parquet
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_mimiciv_v2/data/validation_sae.parquet
/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/sae_mimiciv_v2/data/test_sae.parquet
